In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# ── Cell 1 : imports + load data ─────────────────────────────────────────────
import numpy as np
import pandas as pd
import xarray as xr
from scipy import ndimage
from scipy import stats
import subprocess
import glob, os, warnings
warnings.filterwarnings("ignore")

# install mann-kendall
subprocess.run(['pip', 'install', 'pymannkendall', '--quiet'], check=True)
import pymannkendall as mk

# install regionmask
subprocess.run(['pip', 'install', 'regionmask', '--quiet'], check=True)
import regionmask

print("imports done ✓")

# ── load ERA5 data ──
DATA_DIR = '/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday'
nc_files  = sorted(glob.glob(f'{DATA_DIR}/**/*.nc', recursive=True))
print(f"found {len(nc_files)} files")

ds = xr.open_mfdataset(nc_files, combine='by_coords',
                        engine='h5netcdf', chunks=None)

data_precip = ds['tp'].values.astype('float32')
lat         = ds['latitude'].values.astype('float32')
lon         = ds['longitude'].values.astype('float32')
time        = ds['time'].values
times_pd    = pd.to_datetime(time)

print(f"shape : {data_precip.shape}")
print(f"time  : {str(times_pd[0])[:10]} → {str(times_pd[-1])[:10]}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 2.9 MB/s eta 0:00:00
imports done ✓
found 46 files
shape : (5612, 141, 161)
time  : 1979-06-01 → 2024-09-30


In [6]:
# ── CELL 2 : India mask + threshold + exceedance mask ─────────────────────────
import numpy as np
import pandas as pd
import xarray as xr
from scipy import ndimage
import subprocess, glob, os, warnings
warnings.filterwarnings("ignore")

subprocess.run(['pip', 'install', 'pymannkendall', '--quiet'], check=True)
subprocess.run(['pip', 'install', 'regionmask',    '--quiet'], check=True)
import regionmask

RAINY_DAY_MIN = 1.0
PERCENTILE    = 99.0
FLOOR_MM      = 50.0

countries  = regionmask.defined_regions.natural_earth_v5_0_0.countries_110
india_raw  = countries.mask(lon, lat)
india_mask = (india_raw.values == 98)
print(f"India cells : {india_mask.sum()} / {141*161}")

data_masked = np.where(
    (data_precip > RAINY_DAY_MIN) & india_mask[np.newaxis, :, :],
    data_precip, np.nan
).astype('float32')

jjas_mask          = (times_pd.month >= 6) & (times_pd.month <= 9)
jjas_data          = data_masked[jjas_mask, :, :]
per_grid_99        = np.nanpercentile(jjas_data, PERCENTILE, axis=0).astype('float32')
per_grid_threshold = np.maximum(per_grid_99, FLOOR_MM).astype('float32')
per_grid_threshold[~india_mask] = 999.0

data_precip_safe = np.where(
    np.isnan(data_precip), 0.0, data_precip
).astype('float32')
threshold_3d = np.broadcast_to(
    per_grid_threshold[np.newaxis, :, :], data_precip.shape
).copy().astype('float32')
c = (data_precip_safe > threshold_3d)

print(f"exceedance rate : {c.mean()*100:.3f}%")
print("Cell 2 done ✓")

India cells : 4452 / 22701
exceedance rate : 0.114%
Cell 2 done ✓


In [7]:
# ── CELL 3 : 8-connectivity labeling ──────────────────────────────────────────
struct8 = ndimage.generate_binary_structure(2, 2)

n_days = data_precip.shape[0]
n_lat  = len(lat)
n_lon  = len(lon)

Label8  = np.zeros((n_days, n_lat, n_lon), dtype='int32')
NE8_raw = np.zeros(n_days, dtype='int32')
NT8_raw = np.zeros(n_days, dtype='int32')

for k in range(n_days):
    img        = c[k].astype('int8')
    NT8_raw[k] = int(img.sum())
    lab, nr    = ndimage.label(img, structure=struct8)
    Label8[k]  = lab
    NE8_raw[k] = nr
    if k % 500 == 0:
        print(f"  labeled day {k:4d}/{n_days}")

print(f"\nmean NE = {NE8_raw.mean():.3f}  (expected ~2.434)")
print(f"mean NT = {NT8_raw.mean():.3f}  (expected ~25.93)")
print("Cell 3 done ✓")

  labeled day    0/5612
  labeled day  500/5612
  labeled day 1000/5612
  labeled day 1500/5612
  labeled day 2000/5612
  labeled day 2500/5612
  labeled day 3000/5612
  labeled day 3500/5612
  labeled day 4000/5612
  labeled day 4500/5612
  labeled day 5000/5612
  labeled day 5500/5612

mean NE = 2.434  (expected ~2.434)
mean NT = 25.929  (expected ~25.93)
Cell 3 done ✓


In [9]:
# ── CELL 4 : Hyperparameters + global variables ────────────────────────────────
MIN_TRACK_CELLS   = 5
MIN_TREND_CELLS   = 16
MUTUAL_STRUCTURAL = 0.10
MUTUAL_CONT       = 0.05
MIN_SHARED_CELLS  = 2      # NEW — guards single-cell boundary touches
SEARCH_DEG        = 2.5
SIZE_RATIO_LO     = 0.4
SIZE_RATIO_HI     = 3.0
GAP_TOLERANCE     = 1
DOMAIN_EXIT_FRAC  = 0.50

OUT = '/kaggle/working/'
os.makedirs(OUT, exist_ok=True)

track_stats_rows        = []
merge_event_rows        = []
split_event_rows        = []
global_track_num        = 0
global_merge_id         = 0
global_split_id         = 0
dormancy_events_counter = 0
active_tracks           = {}
births_per_date         = {}

first_jjas_days = {
    pd.Timestamp(f'{yr}-06-01').strftime('%Y%m%d')
    for yr in range(1979, 2025)
}

print("Cell 4 done ✓")
print(f"MIN_SHARED_CELLS = {MIN_SHARED_CELLS}")

Cell 4 done ✓
MIN_SHARED_CELLS = 2


In [10]:
# ── CELL 5 : Helper functions ──────────────────────────────────────────────────
def jjas_day_num(ts):
    return int((ts - pd.Timestamp(f'{ts.year}-06-01')).days + 1)


def get_objects(day_idx, min_cells=MIN_TRACK_CELLS):
    labeled = Label8[day_idx]
    n_obj   = NE8_raw[day_idx]
    objs    = {}
    for lbl in range(1, n_obj + 1):
        mask = (labeled == lbl)
        sz   = int(mask.sum())
        if sz < min_cells:
            continue
        rows_, cols_ = np.where(mask)
        clat    = float(lat[rows_].mean())
        clon    = float(lon[cols_].mean())
        outside = int((mask & ~india_mask).sum())
        objs[lbl] = {'size': sz, 'centroid': (clat, clon), 'outside': outside}
    return objs


def compute_overlaps(day_t, day_t1, objs_t, objs_t1):
    """
    Returns {(lbl_i, lbl_j): (fwd, bwd, shared, mutual)}
    fwd    = shared / size(i)   — fraction of T object going into T+1 object
    bwd    = shared / size(j)   — fraction of T+1 object coming from T object
    mutual = min(fwd, bwd)      — used for dominant links and continuation
    No threshold applied here.
    """
    lab_t  = Label8[day_t]
    lab_t1 = Label8[day_t1]
    pairs  = {}
    for lbl_i, obj_i in objs_t.items():
        mask_i         = (lab_t == lbl_i)
        overlap_labels = np.unique(lab_t1[mask_i])
        overlap_labels = overlap_labels[overlap_labels > 0]
        for lbl_j in overlap_labels:
            if lbl_j not in objs_t1:
                continue
            shared = int((mask_i & (lab_t1 == lbl_j)).sum())
            if shared == 0:
                continue
            fwd    = shared / obj_i['size']
            bwd    = shared / objs_t1[lbl_j]['size']
            mutual = min(fwd, bwd)
            pairs[(lbl_i, lbl_j)] = (fwd, bwd, shared, mutual)
    return pairs


print("Cell 5 done ✓")

Cell 5 done ✓


In [11]:
# ── CELL 6 : Track management functions ───────────────────────────────────────
def new_track(day_idx, lbl, obj, start_type, split_from=-1):
    global global_track_num
    global_track_num += 1
    tnum     = global_track_num
    ts       = times_pd[day_idx]
    date_str = ts.strftime('%Y%m%d')
    births_per_date[date_str] = births_per_date.get(date_str, 0) + 1
    tid = f'{date_str}_{births_per_date[date_str]:03d}'
    clat, clon = obj['centroid']
    row = {
        'track_num'        : tnum,
        'track_id'         : tid,
        'date'             : date_str,
        'year'             : ts.year,
        'jjas_day'         : jjas_day_num(ts),
        'day_of_track'     : 0,
        'size'             : obj['size'],
        'centroid_lat'     : round(clat, 3),
        'centroid_lon'     : round(clon, 3),
        'dormant'          : False,
        'merge_event'      : False,
        'split_event'      : False,
        'start_type'       : start_type,
        'end_type'         : None,
        'duration'         : None,
        'split_from_track' : split_from,
        'merge_into_track' : -1,
        'day_lbl'          : lbl,
    }
    active_tracks[tnum] = {
        'track_id'       : tid,
        'birth_date'     : date_str,
        'last_day'       : day_idx,
        'last_lbl'       : lbl,
        'last_centroid'  : obj['centroid'],
        'last_size'      : obj['size'],
        'dormant_days'   : 0,
        'start_type'     : start_type,
        'duration'       : 1,
        'split_from'     : split_from,
        'n_merge_events' : 0,
        'n_split_events' : 0,
        'daily_rows'     : [row],
    }
    return tnum


def append_track_day(tnum, day_idx, lbl, obj,
                     dormant=False, merge_event=False, split_event=False):
    tr = active_tracks[tnum]
    tr['duration'] += 1
    ts         = times_pd[day_idx]
    sz         = obj['size']     if not dormant else 0
    clat, clon = obj['centroid'] if not dormant else (None, None)
    if not dormant:
        tr['dormant_days'] = 0
        if merge_event: tr['n_merge_events'] += 1
        if split_event: tr['n_split_events'] += 1
    row = {
        'track_num'        : tnum,
        'track_id'         : tr['track_id'],
        'date'             : ts.strftime('%Y%m%d'),
        'year'             : ts.year,
        'jjas_day'         : jjas_day_num(ts),
        'day_of_track'     : tr['duration'] - 1,
        'size'             : sz,
        'centroid_lat'     : round(clat, 3) if clat is not None else None,
        'centroid_lon'     : round(clon, 3) if clon is not None else None,
        'dormant'          : dormant,
        'merge_event'      : merge_event,
        'split_event'      : split_event,
        'start_type'       : tr['start_type'],
        'end_type'         : None,
        'duration'         : None,
        'split_from_track' : tr['split_from'],
        'merge_into_track' : -1,
        'day_lbl'          : lbl if not dormant else -1,
    }
    tr['daily_rows'].append(row)


def terminate_track(tnum, end_type, merge_into=-1):
    global track_stats_rows
    if tnum not in active_tracks:
        return
    tr  = active_tracks[tnum]
    dur = tr['duration']
    rows = tr['daily_rows']
    if end_type == 'natural_death':
        while rows and rows[-1]['dormant']:
            rows.pop()
            dur -= 1
    for r in rows:
        r['end_type']  = end_type
        r['duration']  = dur
        if merge_into > 0:
            r['merge_into_track'] = merge_into
    track_stats_rows.extend(rows)
    del active_tracks[tnum]


print("Cell 6 done ✓")

Cell 6 done ✓


In [12]:
# ── CELL 7 : Main tracking loop ────────────────────────────────────────────────
import time as _time

print("starting main tracking loop …")
_t0 = _time.time()

# ── PRE-LOOP: birth Jun 1 1979 ────────────────────────────────────────────────
objs_day0 = get_objects(0)
for lbl_j, obj_j in objs_day0.items():
    tnum_new = new_track(0, lbl_j, obj_j, 'season_start')
    if obj_j['outside'] / obj_j['size'] > DOMAIN_EXIT_FRAC:
        terminate_track(tnum_new, 'domain_exit')
print(f"pre-loop: Jun 1 1979 — {len(objs_day0)} season_start objects born")

for day_idx in range(n_days - 1):

    ts_today = times_pd[day_idx]
    ts_next  = times_pd[day_idx + 1]

    # ── Sep 30 hard termination + birth next Jun 1 ───────────────────────────
    if ts_today.month == 9 and ts_today.day == 30:
        for tnum in list(active_tracks.keys()):
            terminate_track(tnum, 'season_end')
        if day_idx + 1 < n_days:
            objs_jun1 = get_objects(day_idx + 1)
            for lbl_j, obj_j in objs_jun1.items():
                tnum_new = new_track(day_idx + 1, lbl_j, obj_j, 'season_start')
                if obj_j['outside'] / obj_j['size'] > DOMAIN_EXIT_FRAC:
                    terminate_track(tnum_new, 'domain_exit')
        continue

    objs_today = get_objects(day_idx)
    objs_next  = get_objects(day_idx + 1)

    # ── Bug 2 fix: two separate early-exit cases ──────────────────────────────
    if len(objs_next) == 0:
        for tnum in list(active_tracks.keys()):
            tr = active_tracks.get(tnum)
            if tr is None:
                continue
            if tr['dormant_days'] < GAP_TOLERANCE:
                tr['dormant_days'] += 1
                dormancy_events_counter += 1
                dummy = {'size': 0, 'centroid': tr['last_centroid']}
                append_track_day(tnum, day_idx + 1, -1, dummy, dormant=True)
            else:
                terminate_track(tnum, 'natural_death')
        continue

    if len(objs_today) == 0:
        for tnum in list(active_tracks.keys()):
            tr = active_tracks.get(tnum)
            if tr is None:
                continue
            if tr['dormant_days'] < GAP_TOLERANCE:
                tr['dormant_days'] += 1
                dormancy_events_counter += 1
                dummy = {'size': 0, 'centroid': tr['last_centroid']}
                append_track_day(tnum, day_idx + 1, -1, dummy, dormant=True)
            else:
                terminate_track(tnum, 'natural_death')
        for lbl_j, obj_j in objs_next.items():
            tnum_new = new_track(day_idx + 1, lbl_j, obj_j, 'natural_birth')
            if obj_j['outside'] / obj_j['size'] > DOMAIN_EXIT_FRAC:
                terminate_track(tnum_new, 'domain_exit')
        continue

    # ── STEP 1: compute all overlaps ─────────────────────────────────────────
    pairs = compute_overlaps(day_idx, day_idx + 1, objs_today, objs_next)

    # fwd_links[lbl_i] = list of (mutual, fwd, bwd, shared, lbl_j)
    # bwd_links[lbl_j] = list of (mutual, fwd, bwd, shared, lbl_i)
    fwd_links = {}
    bwd_links = {}
    for (lbl_i, lbl_j), (fwd, bwd, shared, mutual) in pairs.items():
        fwd_links.setdefault(lbl_i, []).append((mutual, fwd, bwd, shared, lbl_j))
        bwd_links.setdefault(lbl_j, []).append((mutual, fwd, bwd, shared, lbl_i))

    # lbl_to_track: today's label → track_num, only for active tracks
    lbl_to_track = {
        tr['last_lbl']: tnum
        for tnum, tr in active_tracks.items()
        if tr['last_day'] == day_idx and tr['dormant_days'] == 0
    }

    handled_today = set()
    handled_next  = set()

    # ─────────────────────────────────────────────────────────────────────────
    # STEP 2: MERGE DETECTION
    #
    # For each T+1 object j with 2+ T parents:
    #   DOMINANT parent: highest mutual = min(fwd,bwd) >= MUTUAL_STRUCTURAL
    #                    AND must be a tracked system
    #   SECOND  parent: fwd = shared/size(parent) >= MUTUAL_STRUCTURAL
    #                   AND shared >= MIN_SHARED_CELLS
    #                   AND must be a tracked system
    #
    # Physically:
    #   dominant uses mutual — both objects must agree on the primary relationship
    #   second   uses fwd   — measures how committed the second parent is
    #                         not contaminated by how large C grew
    # ─────────────────────────────────────────────────────────────────────────
    for lbl_j in sorted(objs_next.keys()):
        if lbl_j in handled_next:
            continue

        candidates = bwd_links.get(lbl_j, [])
        if len(candidates) < 2:
            continue

        # sort by mutual score descending to find dominant first
        candidates_sorted = sorted(candidates, key=lambda x: x[0], reverse=True)

        # find dominant parent
        dominant_lbl  = None
        dominant_tnum = None
        dominant_mut  = None
        dominant_fwd  = None
        for (mutual, fwd, bwd, shared, lbl_i) in candidates_sorted:
            if mutual < MUTUAL_STRUCTURAL:
                break
            if lbl_i in handled_today:
                continue
            if lbl_i not in lbl_to_track:
                continue
            dominant_lbl  = lbl_i
            dominant_tnum = lbl_to_track[lbl_i]
            dominant_mut  = mutual
            dominant_fwd  = fwd
            break

        if dominant_lbl is None:
            continue

        # find second parents using FWD score
        second_parents = []
        for (mutual, fwd, bwd, shared, lbl_i) in candidates_sorted:
            if lbl_i == dominant_lbl:
                continue
            if lbl_i in handled_today:
                continue
            if lbl_i not in lbl_to_track:
                continue
            # NEW RULE: fwd = shared/size(parent)
            if fwd >= MUTUAL_STRUCTURAL and shared >= MIN_SHARED_CELLS:
                second_parents.append((fwd, shared, lbl_i, lbl_to_track[lbl_i]))

        if len(second_parents) == 0:
            continue

        # take best second parent by fwd score
        second_parents.sort(reverse=True)
        abs_fwd, abs_shared, absorbed_lbl, absorbed_tnum = second_parents[0]

        # ── MERGE DECLARED ────────────────────────────────────────────────────
        global_merge_id += 1
        obj_j   = objs_next[lbl_j]
        obj_dom = objs_today[dominant_lbl]
        obj_abs = objs_today[absorbed_lbl]

        inside_CI  = (15.0 <= obj_j['centroid'][0] <= 25.0 and
                      75.0 <= obj_j['centroid'][1] <= 85.0)
        merge_type = ('absorption' if obj_abs['size'] < MIN_TREND_CELLS
                      else 'merge')

        merge_event_rows.append({
            'merge_id'             : global_merge_id,
            'date'                 : ts_next.strftime('%Y%m%d'),
            'year'                 : ts_next.year,
            'jjas_day'             : jjas_day_num(ts_next),
            'dominant_track'       : dominant_tnum,
            'absorbed_track'       : absorbed_tnum,
            'n_absorbed'           : 1,
            'dominant_size_before' : obj_dom['size'],
            'absorbed_size_before' : obj_abs['size'],
            'merged_size_after'    : obj_j['size'],
            'mutual_dominant'      : dominant_mut,
            'mutual_absorbed'      : abs_fwd,
            'absorbed_fwd'         : abs_fwd,
            'absorbed_bwd'         : abs_shared / obj_j['size'],
            'centroid_lat'         : round(obj_j['centroid'][0], 3),
            'centroid_lon'         : round(obj_j['centroid'][1], 3),
            'inside_CI'            : inside_CI,
            'merge_type'           : merge_type,
        })

        append_track_day(dominant_tnum, day_idx + 1, lbl_j, obj_j,
                         merge_event=True)
        active_tracks[dominant_tnum]['last_day']      = day_idx + 1
        active_tracks[dominant_tnum]['last_lbl']      = lbl_j
        active_tracks[dominant_tnum]['last_centroid'] = obj_j['centroid']
        active_tracks[dominant_tnum]['last_size']     = obj_j['size']

        terminate_track(absorbed_tnum, 'merge_death', merge_into=dominant_tnum)

        handled_today.add(dominant_lbl)
        handled_today.add(absorbed_lbl)
        handled_next.add(lbl_j)

    # ─────────────────────────────────────────────────────────────────────────
    # STEP 3: SPLIT DETECTION
    #
    # For each tracked T object i with 2+ T+1 daughters:
    #   DOMINANT daughter: highest mutual = min(fwd,bwd) >= MUTUAL_STRUCTURAL
    #   SECOND  daughter:  bwd = shared/size(daughter) >= MUTUAL_STRUCTURAL
    #                      AND shared >= MIN_SHARED_CELLS
    #
    # Physically:
    #   dominant uses mutual — both objects must agree on the primary relationship
    #   second   uses bwd   — measures how much of the daughter came from parent
    #                         not contaminated by how large the parent is
    # ─────────────────────────────────────────────────────────────────────────
    for lbl_i in sorted(objs_today.keys()):
        if lbl_i in handled_today:
            continue
        if lbl_i not in lbl_to_track:
            continue

        candidates = fwd_links.get(lbl_i, [])
        if len(candidates) < 2:
            continue

        candidates_sorted = sorted(candidates, key=lambda x: x[0], reverse=True)

        # find dominant daughter
        dominant_lbl_j = None
        dominant_mut   = None
        dominant_fwd   = None
        dominant_bwd   = None
        for (mutual, fwd, bwd, shared, lbl_j) in candidates_sorted:
            if mutual < MUTUAL_STRUCTURAL:
                break
            if lbl_j in handled_next:
                continue
            dominant_lbl_j = lbl_j
            dominant_mut   = mutual
            dominant_fwd   = fwd
            dominant_bwd   = bwd
            break

        if dominant_lbl_j is None:
            continue

        # find second daughters using BWD score
        second_daughters = []
        for (mutual, fwd, bwd, shared, lbl_j) in candidates_sorted:
            if lbl_j == dominant_lbl_j:
                continue
            if lbl_j in handled_next:
                continue
            # NEW RULE: bwd = shared/size(daughter)
            if bwd >= MUTUAL_STRUCTURAL and shared >= MIN_SHARED_CELLS:
                second_daughters.append((bwd, shared, lbl_j))

        if len(second_daughters) == 0:
            continue

        # take best second daughter by bwd score
        second_daughters.sort(reverse=True)
        child_bwd, child_shared, child_lbl_j = second_daughters[0]

        parent_tnum = lbl_to_track[lbl_i]
        obj_i       = objs_today[lbl_i]
        obj_dom_j   = objs_next[dominant_lbl_j]
        obj_child_j = objs_next[child_lbl_j]

        # ensure dominant is the larger daughter
        # if child is larger, swap — and retrieve scores correctly from pairs dict
        if obj_child_j['size'] > obj_dom_j['size']:
            dominant_lbl_j, child_lbl_j   = child_lbl_j, dominant_lbl_j
            obj_dom_j,      obj_child_j   = obj_child_j, obj_dom_j
            # get correct scores for the new dominant and new child from pairs dict
            d_entry = pairs.get((lbl_i, dominant_lbl_j))
            c_entry = pairs.get((lbl_i, child_lbl_j))
            dominant_mut = min(d_entry[0], d_entry[1]) if d_entry else dominant_mut
            dominant_fwd = d_entry[0]                  if d_entry else dominant_fwd
            child_bwd    = c_entry[1]                  if c_entry else child_bwd
            child_shared = c_entry[2]                  if c_entry else child_shared

        # lost_fraction: fraction of parent's cells unaccounted for
        # = 1 - sum of fwd fractions to all daughters found
        fwd_to_dominant = dominant_fwd
        fwd_to_child    = child_shared / obj_i['size'] if obj_i['size'] > 0 else 0.0
        lost_fraction   = max(0.0, 1.0 - fwd_to_dominant - fwd_to_child)

        inside_CI = (15.0 <= obj_i['centroid'][0] <= 25.0 and
                     75.0 <= obj_i['centroid'][1] <= 85.0)

        # ── SPLIT DECLARED ────────────────────────────────────────────────────
        global_split_id += 1

        split_event_rows.append({
            'split_id'          : global_split_id,
            'date'              : ts_next.strftime('%Y%m%d'),
            'year'              : ts_next.year,
            'jjas_day'          : jjas_day_num(ts_next),
            'parent_track'      : parent_tnum,
            'dominant_daughter' : parent_tnum,
            'child_track_1'     : -1,
            'n_children'        : 1,
            'parent_size'       : obj_i['size'],
            'dominant_size'     : obj_dom_j['size'],
            'child_size_1'      : obj_child_j['size'],
            'mutual_dominant'   : dominant_mut,
            'child_bwd'         : child_bwd,
            'child_fwd'         : fwd_to_child,
            'size_asymmetry'    : (obj_dom_j['size'] /
                                   (obj_dom_j['size'] + obj_child_j['size'])),
            'lost_fraction'     : lost_fraction,
            'centroid_lat'      : round(obj_i['centroid'][0], 3),
            'centroid_lon'      : round(obj_i['centroid'][1], 3),
            'inside_CI'         : inside_CI,
        })

        # parent track continues as dominant daughter
        append_track_day(parent_tnum, day_idx + 1, dominant_lbl_j, obj_dom_j,
                         split_event=True)
        active_tracks[parent_tnum]['last_day']      = day_idx + 1
        active_tracks[parent_tnum]['last_lbl']      = dominant_lbl_j
        active_tracks[parent_tnum]['last_centroid'] = obj_dom_j['centroid']
        active_tracks[parent_tnum]['last_size']     = obj_dom_j['size']

        # child born as split_birth
        child_tnum = new_track(day_idx + 1, child_lbl_j, obj_child_j,
                               'split_birth', split_from=parent_tnum)
        split_event_rows[-1]['child_track_1'] = child_tnum

        if obj_child_j['outside'] / obj_child_j['size'] > DOMAIN_EXIT_FRAC:
            terminate_track(child_tnum, 'domain_exit')

        handled_today.add(lbl_i)
        handled_next.add(dominant_lbl_j)
        handled_next.add(child_lbl_j)

    # ── STEP 4: CONTINUATION — unchanged, uses mutual = min(fwd,bwd) ─────────
    cont_candidates = []
    for (lbl_i, lbl_j), (fwd, bwd, shared, mutual) in pairs.items():
        if lbl_i in handled_today:
            continue
        if lbl_j in handled_next:
            continue
        if lbl_i not in lbl_to_track:
            continue
        if mutual >= MUTUAL_CONT:
            cont_candidates.append((mutual, lbl_i, lbl_j))

    cont_candidates.sort(reverse=True)
    for (mutual, lbl_i, lbl_j) in cont_candidates:
        if lbl_i in handled_today or lbl_j in handled_next:
            continue
        tnum  = lbl_to_track[lbl_i]
        obj_j = objs_next[lbl_j]
        append_track_day(tnum, day_idx + 1, lbl_j, obj_j)
        active_tracks[tnum]['last_day']      = day_idx + 1
        active_tracks[tnum]['last_lbl']      = lbl_j
        active_tracks[tnum]['last_centroid'] = obj_j['centroid']
        active_tracks[tnum]['last_size']     = obj_j['size']
        handled_today.add(lbl_i)
        handled_next.add(lbl_j)

    # ── STEP 5: FALLBACK — unchanged ─────────────────────────────────────────
    for tnum, tr in list(active_tracks.items()):
        if tr['last_day'] != day_idx:
            continue
        if tr['dormant_days'] != 0:
            continue
        lbl_i = tr['last_lbl']
        if lbl_i in handled_today:
            continue
        clat_i, clon_i = tr['last_centroid']
        sz_i           = tr['last_size']
        best_lbl_j     = None
        best_dist      = float('inf')
        for lbl_j, obj_j in objs_next.items():
            if lbl_j in handled_next:
                continue
            clat_j, clon_j = obj_j['centroid']
            dist = ((clat_j - clat_i)**2 + (clon_j - clon_i)**2)**0.5
            if dist > SEARCH_DEG:
                continue
            ratio = obj_j['size'] / sz_i if sz_i > 0 else 999
            if ratio < SIZE_RATIO_LO or ratio > SIZE_RATIO_HI:
                continue
            if dist < best_dist:
                best_dist  = dist
                best_lbl_j = lbl_j
        if best_lbl_j is not None:
            obj_j = objs_next[best_lbl_j]
            append_track_day(tnum, day_idx + 1, best_lbl_j, obj_j)
            active_tracks[tnum]['last_day']      = day_idx + 1
            active_tracks[tnum]['last_lbl']      = best_lbl_j
            active_tracks[tnum]['last_centroid'] = obj_j['centroid']
            active_tracks[tnum]['last_size']     = obj_j['size']
            handled_today.add(lbl_i)
            handled_next.add(best_lbl_j)

    # ── STEP 6: DORMANCY — unchanged ─────────────────────────────────────────
    for tnum, tr in list(active_tracks.items()):
        if tr['last_day'] != day_idx:
            continue
        if tr['dormant_days'] != 0:
            continue
        lbl_i = tr['last_lbl']
        if lbl_i in handled_today:
            continue
        if tr['dormant_days'] < GAP_TOLERANCE:
            tr['dormant_days'] += 1
            dormancy_events_counter += 1
            dummy = {'size': 0, 'centroid': tr['last_centroid']}
            append_track_day(tnum, day_idx + 1, -1, dummy, dormant=True)
        else:
            terminate_track(tnum, 'natural_death')

    # ── STEP 7: BIRTHS — unchanged ────────────────────────────────────────────
    for lbl_j, obj_j in objs_next.items():
        if lbl_j in handled_next:
            continue
        tnum_new = new_track(day_idx + 1, lbl_j, obj_j, 'natural_birth')
        if obj_j['outside'] / obj_j['size'] > DOMAIN_EXIT_FRAC:
            terminate_track(tnum_new, 'domain_exit')

    if day_idx % 500 == 0 and day_idx > 0:
        print(f"  day {day_idx}/{n_days-1}  "
              f"active={len(active_tracks):4d}  "
              f"rows={len(track_stats_rows):6d}  "
              f"elapsed={_time.time()-_t0:.1f}s")

# terminate remaining tracks
for tnum in list(active_tracks.keys()):
    terminate_track(tnum, 'natural_death')

print(f"\ntracking loop complete — {_time.time()-_t0:.1f}s")
print(f"total track_stats_rows  : {len(track_stats_rows):,}")
print(f"total merge_event_rows  : {len(merge_event_rows):,}")
print(f"total split_event_rows  : {len(split_event_rows):,}")
print(f"dormancy events         : {dormancy_events_counter:,}")

starting main tracking loop …
pre-loop: Jun 1 1979 — 0 season_start objects born
  day 1500/5611  active=   6  rows=  1655  elapsed=0.9s
  day 2000/5611  active=  11  rows=  2263  elapsed=1.2s
  day 2500/5611  active=  14  rows=  2926  elapsed=1.6s
  day 4000/5611  active=   3  rows=  4660  elapsed=2.5s
  day 4500/5611  active=   2  rows=  5232  elapsed=2.8s
  day 5000/5611  active=   6  rows=  5788  elapsed=3.1s

tracking loop complete — 3.5s
total track_stats_rows  : 6,614
total merge_event_rows  : 45
total split_event_rows  : 37
dormancy events         : 4,262


In [13]:
# ── CELL 8 : Build DataFrames + summary ───────────────────────────────────────
df_stats = pd.DataFrame(track_stats_rows)
df_merge = pd.DataFrame(merge_event_rows) if merge_event_rows else pd.DataFrame()
df_split = pd.DataFrame(split_event_rows) if split_event_rows else pd.DataFrame()

summary_rows = []
for tnum, grp in df_stats.groupby('track_num'):
    active_rows = grp[~grp['dormant']]
    first_row   = grp.iloc[0]
    last_row    = grp.iloc[-1]
    summary_rows.append({
        'track_num'        : tnum,
        'track_id'         : first_row['track_id'],
        'birth_date'       : first_row['date'],
        'death_date'       : last_row['date'],
        'duration'         : int(last_row['duration']) if pd.notna(last_row['duration']) else len(grp),
        'peak_size'        : int(active_rows['size'].max()) if len(active_rows) > 0 else 0,
        'mean_size'        : round(float(active_rows['size'].mean()), 2) if len(active_rows) > 0 else 0.0,
        'birth_lat'        : first_row['centroid_lat'],
        'birth_lon'        : first_row['centroid_lon'],
        'start_type'       : first_row['start_type'],
        'end_type'         : last_row['end_type'],
        'split_from_track' : first_row['split_from_track'],
        'merge_into_track' : last_row['merge_into_track'],
        'n_merge_events'   : int(grp['merge_event'].sum()),
        'n_split_events'   : int(grp['split_event'].sum()),
    })

df_summary = pd.DataFrame(summary_rows)

print("DataFrames built ✓")
print(f"df_stats   : {df_stats.shape}")
print(f"df_summary : {df_summary.shape}")
print(f"df_merge   : {df_merge.shape}")
print(f"df_split   : {df_split.shape}")

print('\n' + '=' * 58)
print('CP5 NEW ALGORITHM — SUMMARY')
print('=' * 58)
print(f"total tracks       : {df_summary.shape[0]:,}")
print(f"total merges       : {len(df_merge):,}   (old MIN: 23)")
print(f"total splits       : {len(df_split):,}   (old MIN: 16)")
print(f"dormancy events    : {dormancy_events_counter:,}")

print("\nStart types:")
print(df_summary['start_type'].value_counts().to_string())
print("\nEnd types:")
print(df_summary['end_type'].value_counts().to_string())

print("\nDuration by size class:")
for label, lo, hi in [('Sub-ref  (5-15)', 5,15),
                       ('Medium  (16-90)', 16,90),
                       ('Large    (>=91)', 91,9999)]:
    g = df_summary[df_summary['peak_size'].between(lo, hi)]
    if len(g) == 0: continue
    print(f"  {label}  n={len(g):,}  "
          f"mean={g['duration'].mean():.2f}d  "
          f"single={(g['duration']==1).mean()*100:.1f}%")

years = np.arange(1979, 2025)
if len(df_merge) > 0:
    yr_m = df_merge.groupby('year').size().reindex(years, fill_value=0)
    print(f"\nMerge/yr : {yr_m.mean():.2f}  (old: 0.50)")
    print(f"  'merge' type      : {(df_merge['merge_type']=='merge').sum()}")
    print(f"  'absorption' type : {(df_merge['merge_type']=='absorption').sum()}")
    gf = (df_merge['merged_size_after'] /
          (df_merge['dominant_size_before'] + df_merge['absorbed_size_before']))
    print(f"  growth factor  mean={gf.mean():.2f}  median={gf.median():.2f}")
    print(f"  absorbed_fwd   mean={df_merge['absorbed_fwd'].mean():.3f}  "
          f"min={df_merge['absorbed_fwd'].min():.3f}")
    print(f"  absorbed_bwd   mean={df_merge['absorbed_bwd'].mean():.3f}  "
          f"min={df_merge['absorbed_bwd'].min():.3f}")

if len(df_split) > 0:
    yr_s = df_split.groupby('year').size().reindex(years, fill_value=0)
    print(f"\nSplit/yr : {yr_s.mean():.2f}  (old: 0.35)")
    print(f"  child_bwd  mean={df_split['child_bwd'].mean():.3f}  "
          f"min={df_split['child_bwd'].min():.3f}")
    print(f"  child_fwd  mean={df_split['child_fwd'].mean():.3f}  "
          f"min={df_split['child_fwd'].min():.3f}")
    print(f"  lost_frac  mean={df_split['lost_fraction'].mean():.3f}")
    print(f"  asymmetry  mean={df_split['size_asymmetry'].mean():.3f}")

DataFrames built ✓
df_stats   : (6614, 18)
df_summary : (4322, 15)
df_merge   : (45, 18)
df_split   : (37, 19)

CP5 NEW ALGORITHM — SUMMARY
total tracks       : 4,322
total merges       : 45   (old MIN: 23)
total splits       : 37   (old MIN: 16)
dormancy events    : 4,262

Start types:
start_type
natural_birth    4267
split_birth        37
season_start       18

End types:
end_type
natural_death    4227
season_end         50
merge_death        45

Duration by size class:
  Sub-ref  (5-15)  n=2,636  mean=1.24d  single=79.7%
  Medium  (16-90)  n=1,580  mean=1.92d  single=40.7%
  Large    (>=91)  n=106  mean=3.04d  single=14.2%

Merge/yr : 0.98  (old: 0.50)
  'merge' type      : 13
  'absorption' type : 32
  growth factor  mean=1.82  median=1.67
  absorbed_fwd   mean=0.597  min=0.148
  absorbed_bwd   mean=0.110  min=0.026

Split/yr : 0.80  (old: 0.35)
  child_bwd  mean=0.690  min=0.133
  child_fwd  mean=0.126  min=0.026
  lost_frac  mean=0.652
  asymmetry  mean=0.696


Some questions

In [15]:
# Diagnostic — what is blocking merges and splits NOW
# Run after Cell 3 (labeling done), no tracking needed

import numpy as np
import pandas as pd

MIN_CELLS      = 5
MIN_SHARED     = 2
THRESH         = 0.10

merge_dom_fwd, merge_dom_bwd, merge_dom_mut = [], [], []
merge_sec_fwd, merge_sec_bwd, merge_sec_mut = [], [], []
split_dom_fwd, split_dom_bwd, split_dom_mut = [], [], []
split_sec_fwd, split_sec_bwd, split_sec_mut = [], [], []

for day_idx in range(0, n_days - 1):
    ts = times_pd[day_idx]
    if ts.month == 9 and ts.day == 30:
        continue
    if NE8_raw[day_idx] == 0 or NE8_raw[day_idx + 1] == 0:
        continue

    lab_t  = Label8[day_idx]
    lab_t1 = Label8[day_idx + 1]

    objs_t  = {l: int((lab_t  == l).sum()) for l in range(1, NE8_raw[day_idx]   + 1) if int((lab_t  == l).sum()) >= MIN_CELLS}
    objs_t1 = {l: int((lab_t1 == l).sum()) for l in range(1, NE8_raw[day_idx+1] + 1) if int((lab_t1 == l).sum()) >= MIN_CELLS}
    if not objs_t or not objs_t1:
        continue

    # build all overlap pairs
    all_pairs = {}
    for lbl_i, sz_i in objs_t.items():
        mask_i = (lab_t == lbl_i)
        for lbl_j in np.unique(lab_t1[mask_i]):
            if lbl_j == 0 or lbl_j not in objs_t1:
                continue
            shared = int((mask_i & (lab_t1 == lbl_j)).sum())
            if shared < MIN_SHARED:
                continue
            sz_j  = objs_t1[lbl_j]
            fwd   = shared / sz_i
            bwd   = shared / sz_j
            mut   = min(fwd, bwd)
            all_pairs[(lbl_i, lbl_j)] = (fwd, bwd, shared, mut)

    # bwd_links[lbl_j] = list of (fwd, bwd, shared, mut, lbl_i)
    bwd_links = {}
    fwd_links = {}
    for (lbl_i, lbl_j), (fwd, bwd, shared, mut) in all_pairs.items():
        bwd_links.setdefault(lbl_j, []).append((fwd, bwd, shared, mut, lbl_i))
        fwd_links.setdefault(lbl_i, []).append((fwd, bwd, shared, mut, lbl_j))

    # ── MERGE candidates ─────────────────────────────────────────────────────
    for lbl_j, parents in bwd_links.items():
        if len(parents) < 2:
            continue
        # sort by fwd descending
        parents_f = sorted(parents, key=lambda x: x[0], reverse=True)
        dom = parents_f[0]
        sec = parents_f[1]
        merge_dom_fwd.append(dom[0]);  merge_dom_bwd.append(dom[1]);  merge_dom_mut.append(dom[3])
        merge_sec_fwd.append(sec[0]);  merge_sec_bwd.append(sec[1]);  merge_sec_mut.append(sec[3])

    # ── SPLIT candidates ──────────────────────────────────────────────────────
    for lbl_i, daughters in fwd_links.items():
        if len(daughters) < 2:
            continue
        # sort by bwd descending
        daughters_b = sorted(daughters, key=lambda x: x[1], reverse=True)
        dom = daughters_b[0]
        sec = daughters_b[1]
        split_dom_fwd.append(dom[0]);  split_dom_bwd.append(dom[1]);  split_dom_mut.append(dom[3])
        split_sec_fwd.append(sec[0]);  split_sec_bwd.append(sec[1]);  split_sec_mut.append(sec[3])

# convert
mdf  = np.array(merge_dom_fwd);  mdb  = np.array(merge_dom_bwd);  mdm  = np.array(merge_dom_mut)
msf  = np.array(merge_sec_fwd);  msb  = np.array(merge_sec_bwd);  msm  = np.array(merge_sec_mut)
sdf  = np.array(split_dom_fwd);  sdb  = np.array(split_dom_bwd);  sdm  = np.array(split_dom_mut)
ssf  = np.array(split_sec_fwd);  ssb  = np.array(split_sec_bwd);  ssm  = np.array(split_sec_mut)

Nm = len(mdf)
Ns = len(sdf)

print(f"Merge candidates (2+ parents, shared>={MIN_SHARED}): {Nm}  (~{Nm/46:.1f}/yr)")
print(f"Split candidates (2+ daughters, shared>={MIN_SHARED}): {Ns}  (~{Ns/46:.1f}/yr)\n")

# ── MERGE ANALYSIS ────────────────────────────────────────────────────────────
print("=" * 60)
print("MERGE — DOMINANT PARENT  (sorted by FWD, top candidate)")
print("=" * 60)
print(f"  FWD  mean={mdf.mean():.3f}  median={np.median(mdf):.3f}  "
      f"% >= {THRESH} : {(mdf>=THRESH).mean()*100:.1f}%")
print(f"  BWD  mean={mdb.mean():.3f}  median={np.median(mdb):.3f}  "
      f"% >= {THRESH} : {(mdb>=THRESH).mean()*100:.1f}%")
print(f"  MUT  mean={mdm.mean():.3f}  median={np.median(mdm):.3f}  "
      f"% >= {THRESH} : {(mdm>=THRESH).mean()*100:.1f}%")
print(f"\n  Bottleneck (what limits MUT):")
print(f"    FWD < BWD (FWD limits): {(mdf<mdb).sum()} ({(mdf<mdb).mean()*100:.1f}%)")
print(f"    BWD < FWD (BWD limits): {(mdb<mdf).sum()} ({(mdb<mdf).mean()*100:.1f}%)")
print(f"\n  If we use FWD only for dominant:")
for t in [0.05, 0.10, 0.15, 0.20]:
    print(f"    FWD >= {t:.2f} : {(mdf>=t).mean()*100:.1f}% pass")

print(f"\n{'─'*60}")
print("MERGE — SECOND PARENT")
print(f"{'─'*60}")
print(f"  FWD  mean={msf.mean():.3f}  median={np.median(msf):.3f}  "
      f"% >= {THRESH} : {(msf>=THRESH).mean()*100:.1f}%")
print(f"  BWD  mean={msb.mean():.3f}  median={np.median(msb):.3f}  "
      f"% >= {THRESH} : {(msb>=THRESH).mean()*100:.1f}%")
print(f"  MUT  mean={msm.mean():.3f}  median={np.median(msm):.3f}  "
      f"% >= {THRESH} : {(msm>=THRESH).mean()*100:.1f}%")
print(f"\n  Bottleneck (what limits MUT):")
print(f"    FWD < BWD (FWD limits): {(msf<msb).sum()} ({(msf<msb).mean()*100:.1f}%)")
print(f"    BWD < FWD (BWD limits): {(msb<msf).sum()} ({(msb<msf).mean()*100:.1f}%)")

print(f"\n  How many candidates pass BOTH FWD(dom) AND FWD(sec) >= {THRESH}?")
both_merge_fwd = ((mdf >= THRESH) & (msf >= THRESH)).sum()
print(f"    Both FWD >= {THRESH} : {both_merge_fwd}  (~{both_merge_fwd/46:.1f}/yr)")

# ── SPLIT ANALYSIS ────────────────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("SPLIT — DOMINANT DAUGHTER  (sorted by BWD, top candidate)")
print("=" * 60)
print(f"  BWD  mean={sdb.mean():.3f}  median={np.median(sdb):.3f}  "
      f"% >= {THRESH} : {(sdb>=THRESH).mean()*100:.1f}%")
print(f"  FWD  mean={sdf.mean():.3f}  median={np.median(sdf):.3f}  "
      f"% >= {THRESH} : {(sdf>=THRESH).mean()*100:.1f}%")
print(f"  MUT  mean={sdm.mean():.3f}  median={np.median(sdm):.3f}  "
      f"% >= {THRESH} : {(sdm>=THRESH).mean()*100:.1f}%")
print(f"\n  Bottleneck (what limits MUT):")
print(f"    BWD < FWD (BWD limits): {(sdb<sdf).sum()} ({(sdb<sdf).mean()*100:.1f}%)")
print(f"    FWD < BWD (FWD limits): {(sdf<sdb).sum()} ({(sdf<sdb).mean()*100:.1f}%)")
print(f"\n  If we use BWD only for dominant:")
for t in [0.05, 0.10, 0.15, 0.20]:
    print(f"    BWD >= {t:.2f} : {(sdb>=t).mean()*100:.1f}% pass")

print(f"\n{'─'*60}")
print("SPLIT — SECOND DAUGHTER")
print(f"{'─'*60}")
print(f"  BWD  mean={ssb.mean():.3f}  median={np.median(ssb):.3f}  "
      f"% >= {THRESH} : {(ssb>=THRESH).mean()*100:.1f}%")
print(f"  FWD  mean={ssf.mean():.3f}  median={np.median(ssf):.3f}  "
      f"% >= {THRESH} : {(ssf>=THRESH).mean()*100:.1f}%")
print(f"  MUT  mean={ssm.mean():.3f}  median={np.median(ssm):.3f}  "
      f"% >= {THRESH} : {(ssm>=THRESH).mean()*100:.1f}%")
print(f"\n  Bottleneck (what limits MUT):")
print(f"    BWD < FWD (BWD limits): {(ssb<ssf).sum()} ({(ssb<ssf).mean()*100:.1f}%)")
print(f"    FWD < BWD (FWD limits): {(ssf<ssb).sum()} ({(ssf<ssb).mean()*100:.1f}%)")

print(f"\n  How many candidates pass BOTH BWD(dom) AND BWD(sec) >= {THRESH}?")
both_split_bwd = ((sdb >= THRESH) & (ssb >= THRESH)).sum()
print(f"    Both BWD >= {THRESH} : {both_split_bwd}  (~{both_split_bwd/46:.1f}/yr)")

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("SUMMARY — what each rule gives at threshold 0.10")
print("=" * 60)
print(f"  Current new algo  : merges={Nm if False else '~0.98'}/yr  splits=~0.80/yr")
print(f"  Merge: both FWD >= 0.10  → {both_merge_fwd}  (~{both_merge_fwd/46:.1f}/yr)")
print(f"  Split: both BWD >= 0.10  → {both_split_bwd}  (~{both_split_bwd/46:.1f}/yr)")
print(f"  Reference target         : ~1.4/yr each")
print(f"\n  Threshold sweep for BOTH FWD (merge) and BOTH BWD (split):")
print(f"  {'thresh':>8}  {'merge/yr':>10}  {'split/yr':>10}")
for t in [0.05, 0.07, 0.08, 0.10, 0.12, 0.15]:
    nm = ((mdf>=t) & (msf>=t)).sum()
    ns = ((sdb>=t) & (ssb>=t)).sum()
    print(f"  {t:>8.2f}  {nm/46:>10.2f}  {ns/46:>10.2f}")

Merge candidates (2+ parents, shared>=2): 64  (~1.4/yr)
Split candidates (2+ daughters, shared>=2): 53  (~1.2/yr)

MERGE — DOMINANT PARENT  (sorted by FWD, top candidate)
  FWD  mean=0.755  median=0.842  % >= 0.1 : 100.0%
  BWD  mean=0.175  median=0.144  % >= 0.1 : 68.8%
  MUT  mean=0.174  median=0.144  % >= 0.1 : 68.8%

  Bottleneck (what limits MUT):
    FWD < BWD (FWD limits): 1 (1.6%)
    BWD < FWD (BWD limits): 63 (98.4%)

  If we use FWD only for dominant:
    FWD >= 0.05 : 100.0% pass
    FWD >= 0.10 : 100.0% pass
    FWD >= 0.15 : 100.0% pass
    FWD >= 0.20 : 98.4% pass

────────────────────────────────────────────────────────────
MERGE — SECOND PARENT
────────────────────────────────────────────────────────────
  FWD  mean=0.384  median=0.372  % >= 0.1 : 84.4%
  BWD  mean=0.140  median=0.103  % >= 0.1 : 50.0%
  MUT  mean=0.126  median=0.093  % >= 0.1 : 46.9%

  Bottleneck (what limits MUT):
    FWD < BWD (FWD limits): 7 (10.9%)
    BWD < FWD (BWD limits): 57 (89.1%)

  How ma